# 01 - Data Exploration

Explore the dataset: speaker statistics, waveform visualization, and spectrogram examples.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torchaudio
import IPython.display as ipd
from src.data.download import prepare_dataset

plt.style.use('seaborn-v0_8-paper')

In [ ]:
# Prepare dataset (creates synthetic if VoxCeleb1 not available)
metadata = prepare_dataset('../data', num_speakers=10, min_utterances=5)
print(f"Total utterances: {len(metadata)}")
print(f"Speakers: {metadata['speaker_id'].nunique()}")
metadata.head(10)

In [ ]:
# Utterances per speaker
fig, ax = plt.subplots(figsize=(10, 5))
counts = metadata.groupby('speaker_id').size().sort_values(ascending=False)
counts.plot(kind='bar', ax=ax)
ax.set_xlabel('Speaker ID')
ax.set_ylabel('Number of Utterances')
ax.set_title('Utterances per Speaker')
plt.tight_layout()
plt.savefig('../results/utterances_per_speaker.png', dpi=300)
plt.show()

In [ ]:
# Duration distribution
fig, ax = plt.subplots(figsize=(8, 5))
metadata['duration_sec'].hist(bins=30, ax=ax, edgecolor='black')
ax.set_xlabel('Duration (seconds)')
ax.set_ylabel('Count')
ax.set_title('Utterance Duration Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize waveforms from different speakers
fig, axes = plt.subplots(3, 1, figsize=(12, 8))
speakers = metadata['speaker_id'].unique()[:3]

for ax, spk in zip(axes, speakers):
    sample = metadata[metadata['speaker_id'] == spk].iloc[0]
    waveform, sr = torchaudio.load(sample['file_path'], backend="soundfile")
    time = np.arange(waveform.shape[1]) / sr
    ax.plot(time, waveform[0].numpy(), linewidth=0.5)
    ax.set_title(f'Speaker: {spk}')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude')

plt.tight_layout()
plt.savefig('../results/waveform_examples.png', dpi=300)
plt.show()

In [ ]:
# Listen to a sample
sample = metadata.iloc[0]
waveform, sr = torchaudio.load(sample['file_path'], backend="soundfile")
print(f"Speaker: {sample['speaker_id']}, Duration: {sample['duration_sec']:.2f}s")
ipd.Audio(waveform[0].numpy(), rate=sr)